# IA Generativa aplicada na área de Dados

##  Importar bibliotecas

In [ ]:
# Importando Bibliotecas
import google.generativeai as genai
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

## Configurar chave de API

In [ ]:
GOOGLE_API_KEY = 'Minha API Key'
genai.configure(api_key=GOOGLE_API_KEY)

# Case prático: Construindo um Chatbot de RH com RAG

Problema de Negócio: A equipe de RH quer um chatbot que responda perguntas de funcionários com base **exclusivamente** nos documentos de políticas da empresa, sem inventar respostas.

## Criando base de conhecimento


In [ ]:
documentos_rh = [
    {"titulo":"Política de Férias","conteudo":"Todo funcionário com mais de 12 meses de contrato tem direito a 30 dias de férias remuneradas. A solicitação deve ser feita com 60 dias de antecedência. É permitido vender até 10 dias de férias."},
    {"titulo": "Política de Home Office", "conteudo": "O modelo de trabalho padrão é híbrido. Para os dias de home office, a empresa oferece uma ajuda de custo de R$ 100,00 mensais para despesas com internet e energia"},
    {"titulo": "Política de Licença Paternidade", "conteudo": "A empresa oferece uma licença paternidade estendida de 30 dias corridos para novos pais, superior aos 5 dias previstos em lei. A licença deve começar na data de nascimento do bebê."}
]

df_docs = pd.DataFrame(documentos_rh)

In [ ]:
df_docs

,titulo,conteudo
0,Política de Férias,Todo funcionário com mais de 12 meses de contr...
1,Política de Home Office,O modelo de trabalho padrão é híbrido. Para os...
2,Política de Licença Paternidade,A empresa oferece uma licença paternidade este...


## A Falha do LLM "Puro"

In [ ]:
model_chat = genai.GenerativeModel('gemini-2.5-flash')
pergunta_usuario = "Quantos dias de licença paternidade a empresa oferece?"
response_sem_reg = model_chat.generate_content(pergunta_usuario)

In [ ]:
print(f"Pergunta: {pergunta_usuario}")
print(f"Resposta do LLM (sem contexto): {response_sem_reg.text}")

Pergunta: Quantos dias de licença paternidade a empresa oferece?
Resposta do LLM (sem contexto): Como inteligência artificial, eu não tenho uma "empresa" nem acesso às políticas de licença paternidade de uma empresa específica.

No Brasil, a licença paternidade **mínima garantida por lei** é de **5 dias corridos**.

No entanto, muitas empresas aderem ao programa **Empresa Cidadã**, que permite estender essa licença para **20 dias corridos**. Empresas que aderem a este programa recebem incentivos fiscais e oferecem um benefício maior aos pais.

Além disso, algumas empresas, como um benefício adicional e diferencial, podem oferecer um período ainda maior de licença paternidade, acima dos 20 dias, por conta própria.

**Para saber quantos dias de licença paternidade *a sua empresa* oferece, você deve consultar:**

1.  **Departamento de Recursos Humanos (RH)** da sua empresa.
2.  **Manual do colaborador** ou políticas internas da empresa.
3.  **Acordo ou Convenção Coletiva de Trabalho** da 

## Implementando o RAG (Retrieval-Augmented Generation)

In [ ]:
# Passo 1: Retrieval - Criar embeddings e encontrar o documento relevante
embedding_pergunta = genai.embed_content(model='models/gemini-embedding-001', content=pergunta_usuario)['embedding']
df_docs['Embedding'] = df_docs['conteudo'].apply(lambda x: genai.embed_content(model='models/gemini-embedding-001', content=x)['embedding'])
df_docs['Similaridade'] = df_docs['Embedding'].apply(lambda x: cosine_similarity([x], [embedding_pergunta])[0][0])

In [ ]:
df_docs

,titulo,conteudo,Embedding,Similaridade
0,Política de Férias,Todo funcionário com mais de 12 meses de contr...,"[-0.014914009, 0.026554843, 0.0071742395, -0.0...",0.705687
1,Política de Home Office,O modelo de trabalho padrão é híbrido. Para os...,"[0.0031186806, 0.018774604, 0.00861908, -0.084...",0.645500
2,Política de Licença Paternidade,A empresa oferece uma licença paternidade este...,"[-0.013877991, 0.027969833, 0.011385356, -0.05...",0.887264


In [ ]:
# Passo 2: Generation - Montar o prompt aumentado e gerar a resposta
documento_relevante = df_docs.sort_values(by='Similaridade', ascending=False).iloc[0]
prompt_com_rag = f"""
Com base APENAS no contexto abaixo, responda à pergunta do usuário. Se a resposta não estiver no contexto, diga "Não encontrei essa informação nos documentos".

**Contexto:**
"{documento_relevante['conteudo']}"

**Pergunta:**
"{pergunta_usuario}"
"""

In [ ]:
response_com_rag = model_chat.generate_content(prompt_com_rag)
print(response_com_rag.text)

A empresa oferece 30 dias de licença paternidade.


## Função para código reutilizável

In [ ]:
def chatbot_rh(pergunta):
  embedding_pergunta = genai.embed_content(model='models/gemini-embedding-001', content=pergunta)['embedding']
  df_docs['Similaridade'] = df_docs['Embedding'].apply(lambda x: cosine_similarity([x], [embedding_pergunta])[0][0])
  documento_relevante = df_docs.sort_values(by='Similaridade', ascending=False).iloc[0]
  prompt_final = f"""
  Com base APENAS no contexto abaixo, responda à pergunta do usuário. Se a resposta não estiver no contexto, diga "Não encontrei essa informação nos documentos".

  **Contexto:**
  "{documento_relevante['conteudo']}"

  **Pergunta:**
  "{pergunta}"
  """
  resposta = model_chat.generate_content(prompt_final)
  return resposta.text

In [ ]:
nova_pergunta = "A empresa oferece ajuda de custo para home office?"

print(f"Pergunta: {nova_pergunta}")
print(f"Resposta: {chatbot_rh(nova_pergunta)}")

Pergunta: A empresa oferece ajuda de custo para home office?
Resposta: Sim, a empresa oferece uma ajuda de custo de R$ 100,00 mensais para despesas com internet e energia nos dias de home office.
